In [ ]:
# ============================================================================
# ML Final Project - Few-Shot Learning
# predict.ipynb: 載入模型預測測試集，輸出 outputs.csv
# ============================================================================

import subprocess
import sys
import os
import json
import pandas as pd
from pathlib import Path

# 自動安裝依賴套件
def install_if_needed(pkg: str):
    try:
        __import__(pkg.split("==")[0])
    except ModuleNotFoundError:
        print(f"[INFO] Installing {pkg}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg])

# 安裝必要套件
for lib in ["open_clip_torch", "timm", "pillow"]:
    install_if_needed(lib)

# 導入套件
import torch
import torch.nn as nn
import timm
import open_clip
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import numpy as np

# ============================================================================
# 設定參數
# ============================================================================
IMG_SIZE = 224
BATCH_SIZE = 32
TEST_DIR = "/test_data/test"
MODEL_PATH = "model.pth"
MODEL_INFO_PATH = "model_info.json"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"[INFO] Using device: {device}")

# ============================================================================
# 模型定義 (與 train.ipynb 相同)
# ============================================================================
class CLIPFineTuner(nn.Module):
    def __init__(self, clip_model, num_classes):
        super(CLIPFineTuner, self).__init__()
        self.clip_model = clip_model
        self.dropout = nn.Dropout(0.1)
        self.classifier = nn.Linear(clip_model.visual.output_dim, num_classes)

    def forward(self, x):
        with torch.no_grad():
            image_features = self.clip_model.encode_image(x)
        image_features = image_features.float()
        image_features = self.dropout(image_features)
        return self.classifier(image_features)

class OptimizedEnsemble(nn.Module):
    def __init__(self, clip_model, timm_model, weights):
        super().__init__()
        self.clip_model = clip_model
        self.timm_model = timm_model
        self.register_buffer('weights', torch.FloatTensor(weights))

    def forward(self, x):
        with torch.no_grad():
            clip_output = torch.softmax(self.clip_model(x), dim=1)
            timm_output = torch.softmax(self.timm_model(x), dim=1)

            # 加權平均
            ensemble_output = (self.weights[0] * clip_output +
                             self.weights[1] * timm_output)

        return ensemble_output

# ============================================================================
# 測試集資料前處理
# ============================================================================
test_transform = transforms.Compose([
    transforms.Resize(IMG_SIZE + 32),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

class TestDataset(Dataset):
    def __init__(self, test_dir, transform=None):
        self.test_dir = Path(test_dir)
        self.transform = transform

        self.image_files = []
        for img_file in self.test_dir.glob("*.jpg"):
            if img_file.name.replace(".jpg", "").isdigit():
                self.image_files.append(img_file)

        # 按照圖片編號排序 (作業要求)
        self.image_files.sort(key=lambda x: int(x.name.replace(".jpg", "")))

        if len(self.image_files) == 0:
            print(f"[WARNING] No test images found in {test_dir}")
            print("[INFO] For testing, you can copy some images from train folder to test folder")

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        img_path = self.image_files[idx]
        image = Image.open(img_path).convert('RGB')

        if self.transform:
            image = self.transform(image)

        # 返回圖片編號 (不含副檔名)
        img_name = img_path.name.replace(".jpg", "")
        return image, img_name

# ============================================================================
# 載入模型和類別名稱
# ============================================================================
def load_model_and_classes():
    # 載入模型資訊
    if not os.path.exists(MODEL_INFO_PATH):
        print(f"[ERROR] Model info file not found: {MODEL_INFO_PATH}")
        print("[INFO] Please run train.ipynb first to generate model_info.json")
        return None, None

    with open(MODEL_INFO_PATH, "r") as f:
        model_info = json.load(f)

    class_names = model_info["class_names"]
    num_classes = model_info["num_classes"]
    best_method = model_info["best_method"]

    print(f"[INFO] Loading {best_method} model...")
    print(f"[INFO] Number of classes: {num_classes}")
    print(f"[INFO] Classes: {', '.join(class_names[:5])}{'...' if num_classes > 5 else ''}")

    # 根據最佳方法載入對應模型
    if best_method == "clip_finetuning":
        # 載入CLIP模型
        clip_model, _, _ = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
        model = CLIPFineTuner(clip_model, num_classes)

    elif best_method == "standard_timm":
        # 載入timm模型
        model = timm.create_model("convnext_tiny", pretrained=False, num_classes=num_classes)

    elif best_method == "optimized_ensemble":
        # 載入ensemble模型
        clip_model, _, _ = open_clip.create_model_and_transforms("ViT-B-32", pretrained="laion2b_s34b_b79k")
        clip_part = CLIPFineTuner(clip_model, num_classes)
        timm_part = timm.create_model("convnext_tiny", pretrained=False, num_classes=num_classes)

        # 獲取ensemble權重
        ensemble_weights = model_info.get("ensemble_weights", [0.5, 0.5])
        model = OptimizedEnsemble(clip_part, timm_part, ensemble_weights)

        print(f"[INFO] Ensemble weights: CLIP={ensemble_weights[0]:.3f}, timm={ensemble_weights[1]:.3f}")

    else:
        print(f"[ERROR] Unknown model method: {best_method}")
        return None, None

    # 載入模型權重
    if not os.path.exists(MODEL_PATH):
        print(f"[ERROR] Model file not found: {MODEL_PATH}")
        print("[INFO] Please run train.ipynb first to generate model.pth")
        return None, None

    try:
        model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
        model = model.to(device)
        model.eval()

        # 顯示模型參數數量 (作業要求)
        total_params = sum(p.numel() for p in model.parameters())
        print(f"[INFO] Model total parameters: {total_params:,} ({total_params/1e6:.1f}M)")

        if total_params > 1000e6:
            print("[WARNING] Model has more than 1000M parameters!")
        else:
            print("[INFO] Model parameter count is within requirement (<1000M)")

        print(f"[INFO] Model loaded successfully: {best_method}")

    except Exception as e:
        print(f"[ERROR] Failed to load model: {e}")
        return None, None

    return model, class_names

# ============================================================================
# 預測函數
# ============================================================================
def predict_test_set(model, test_loader, class_names):
    predictions = []
    image_names = []

    print(f"[INFO] Starting prediction on {len(test_loader.dataset)} test images...")

    with torch.no_grad():
        for batch_idx, (images, img_names) in enumerate(test_loader):
            images = images.to(device)

            # 前向傳播
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)

            # 將預測結果轉換為類別名稱
            for i, pred_idx in enumerate(predicted):
                predictions.append(class_names[pred_idx])
                image_names.append(img_names[i])

            if (batch_idx + 1) % 10 == 0:
                processed = (batch_idx + 1) * len(images)
                total = len(test_loader.dataset)
                print(f"[INFO] Processed {processed} / {total} images ({processed/total*100:.1f}%)")

    return image_names, predictions

# ============================================================================
# 主要預測流程
# ============================================================================
print("[INFO] Loading model and class names...")
model, class_names = load_model_and_classes()

if model is not None and class_names is not None:
    print(f"[INFO] Loading test data from {TEST_DIR}")
    test_dataset = TestDataset(TEST_DIR, transform=test_transform)

    if len(test_dataset) == 0:
        print("[ERROR] No test images found!")
        print("[INFO] Please ensure test images are in /content/test/ folder")
        print("[INFO] Expected format: 5030.jpg, 5351.jpg, etc.")
    else:
        test_loader = DataLoader(
            test_dataset, batch_size=BATCH_SIZE, shuffle=False,
            num_workers=2, pin_memory=True
        )

        print(f"[INFO] Found {len(test_dataset)} test images")

        # 進行預測
        image_names, predictions = predict_test_set(model, test_loader, class_names)

        # 創建結果 DataFrame (按作業要求格式)
        results_df = pd.DataFrame({
            'picture': image_names,
            'class': predictions
        })

        # 按照圖片編號排序 (作業要求數字排序)
        results_df['picture_int'] = results_df['picture'].astype(int)
        results_df = results_df.sort_values('picture_int').drop('picture_int', axis=1)

        # 保存結果為 outputs.csv (作業要求)
        output_file = "outputs.csv"
        results_df.to_csv(output_file, index=False)

        print(f"\n[INFO] Predictions saved to {output_file}")
        print(f"[INFO] Total predictions: {len(results_df)}")

        # 顯示前幾個預測結果
        print("\n[INFO] Sample predictions (first 10 rows):")
        print(results_df.head(10).to_string(index=False))

        # 顯示類別分布
        print(f"\n[INFO] Prediction class distribution:")
        class_counts = results_df['class'].value_counts()
        print(class_counts.head(10).to_string())

        # 作業合規性檢查
        print("\n" + "="*50)
        print("作業要求檢查")
        print("="*50)
        print("✅ 輸出檔案名稱: outputs.csv")
        print("✅ 第一欄: picture (圖片編號)")
        print("✅ 第二欄: class (預測類別)")
        print("✅ 按數字順序排序")
        print("✅ 印出模型參數數量")
        print(f"✅ 預測完成: {len(results_df)} 張圖片")

        # 驗證CSV格式
        if len(results_df.columns) == 2 and list(results_df.columns) == ['picture', 'class']:
            print("✅ CSV格式正確")
        else:
            print("❌ CSV格式有誤")

else:
    print("[ERROR] Failed to load model or class names. Please run train.ipynb first.")
